#### Configuración inicial

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Chunkbench")
CHUNKERS_DIR = PROJECT_DIR / "chunkers"
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
PREPROCESSING_DIR = PROJECT_DIR / "preprocessing"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

Mounted at /content/drive


In [3]:
from document_plans.document_plan_guideline import DOCUMENT_PLAN_GL
from document_plans.document_plan_paper import DOCUMENT_PLAN_PAPER
from document_plans.document_plan_ficha_tecnica import DOCUMENT_PLAN_FC
from scripts.json_router import route_and_chunk_json

from preprocessing.json_structured_chunker import (
    chunk_document as generic_chunk_document,
)

In [4]:
from chunkers.json.JSON_chunker import (
    make_huggingface_token_counter,
)
from transformers import AutoTokenizer

MODEL_NAME = "BAAI/bge-m3"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

bge_m3_token_counter = make_huggingface_token_counter(tokenizer)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [5]:
STRUCTURES_DIR = (
    PROJECT_DIR
    / "document_plans"
    / "structures"
)

PLAN_REGISTRY = {
    "guideline": DOCUMENT_PLAN_GL,
    "paper": DOCUMENT_PLAN_PAPER,
    "ficha_tecnica": DOCUMENT_PLAN_FC,
}

TEMPLATE_PLAN_MAP = {
    "README - GUIDELINE": "guideline",
    "README - PAPER": "paper",
    "README - FICHA TECNICA": "ficha_tecnica",
}

In [8]:
json_path = INPUT_DIR / "GUIDELINE - Leech et al. - 2021 - BSACI 2021 guideline for the management of egg allergy.json"

chunks, routing = route_and_chunk_json(
    json_path=json_path,
    templates_dir=STRUCTURES_DIR,
    plan_registry=PLAN_REGISTRY,
    template_plan_map=TEMPLATE_PLAN_MAP,
    generic_chunk_function=generic_chunk_document,
    threshold=0.50,
    minimum_common_paths=5,
    ignore_single_root=True,
    min_chunk_tokens=100,
    structured_chunker_kwargs={
        "target_tokens": 512,
        "max_tokens": 8192,
        "token_counter": bge_m3_token_counter,
        "indivisible_list_paths": {},
    },
)


In [9]:
print("Método seleccionado:", routing["route"])
print("Molde seleccionado:", routing["selected_template"])
print("Mejor candidato:", routing["best_candidate"])

print(
    "Cobertura:",
    f"{routing['template_coverage']:.2%}",
)

print("Número de chunks:", len(chunks))

Método seleccionado: method_1_hierarchical
Molde seleccionado: README - GUIDELINE
Mejor candidato: README - GUIDELINE
Cobertura: 89.29%
Número de chunks: 65
